# PyStrata workflow

Import the package directly from this repository's `src` directory.

In [1]:
import sys
from pathlib import Path

# Find the repository root whether Jupyter starts in the root or tests/.
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "pystrata").is_dir()
)

src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pystrata
from pystrata.output import KappaOutput
import pykooh

print(f"Using pystrata from: {Path(pystrata.__file__).resolve()}")

Using pystrata from: C:\Users\jimxi\GitHub\pystrata\src\pystrata\__init__.py


In [2]:
import numpy as np
import pyrvt
import pandas as pd

In [3]:
Site_Profile_df = pd.read_excel('data/Meloland_Soil_Profile.xlsx',
                               sheet_name = 'Scaled_Dmin_to_Kappa_0.036')

Layers = []
D_min = []
Vs = []
Thickness = []
Profile_Depth = []
mrd_strains = np.logspace(-6,0,num=20)
ModReduc_data = {}
Damping_data = {}
max_freqs = Site_Profile_df['Max Freq']
wave_fracs = Site_Profile_df['Wave Fraction']

In [4]:

for i, (_, row) in enumerate(Site_Profile_df.iterrows()):

    soil_type = pystrata.site.DarendeliSoilType(unit_wt = row['Unit Weight (kN/m3)'],
                                                plas_index=row['PI'],
                                                ocr=1,
                                                stress_mean=row['Stress (kPa)'],
                                                strains = mrd_strains,
                                                damping_min = row['Scaled_D_min (%)']/100)
    
    Layers.append(pystrata.site.Layer(soil_type,row['Thickness (m)'],row['Velocity (m/s)']))

Layers.append(
            pystrata.site.Layer(
                pystrata.site.SoilType(
                    'Reference Rock',
                    25.9,
                    None,
                    0.01
                ),
                0,
                3500
            )
        )

In [5]:
Site_profile = pystrata.site.Profile(Layers)
discretized_Site_profile = Site_profile.auto_discretize(max_freq = max_freqs,wave_frac=wave_fracs)

In [6]:
def calc_kappa(freq,
               FAS,
               freq_range_for_kappa,
               ko_bandwidth = None,
               show_fit = False):
    
    if ko_bandwidth is not None:
        FAS_for_kappa = pykooh.smooth(
            freq_range_for_kappa,
            freq,
            FAS,
            bw = ko_bandwidth
        )
    else:
        FAS_for_kappa = np.interp(freq_range_for_kappa,freq,FAS)

    coeffs = np.polyfit(freq_range_for_kappa,np.log(FAS_for_kappa),1)
    slope, intercept = coeffs
    kappa = -slope / np.pi

    fit_line = np.exp(slope * freq_range_for_kappa + intercept)

    if show_fit:
        return fit_line
    else:
        return kappa

In [7]:
def Kappa_Correction(freq,FAS,Delta_kappa):

    FAS_adj = np.exp(-np.pi*Delta_kappa*freq)*FAS
    
    return FAS_adj

In [8]:
# Calculation Loop
outputs_freqs = np.logspace(np.log10(0.01),np.log10(50),1000)
RS_freqs = np.logspace(np.log10(0.05),np.log10(50),1000)

Kappa_freqs = outputs_freqs[
    (outputs_freqs >= 10) & (outputs_freqs <= 30)
]

output = pystrata.output.OutputCollection(
    [
        pystrata.output.FourierAmplitudeSpectrumOutput(
            outputs_freqs,
            pystrata.output.OutputLocation("outcrop", index=0),
            None
        ),
        pystrata.output.KappaOutput(
            Kappa_freqs,
            pystrata.output.OutputLocation("outcrop", index=0),
            None
        ),
        pystrata.output.KappaCorrectFourierAmplitudeSpectrumOutput(
            outputs_freqs,
            Kappa_freqs,
            0.039,
            pystrata.output.OutputLocation("outcrop", index=0),
            None
        )
    ] 
    )


motion = pystrata.motion.TimeSeriesMotion.load_at2_file(
    'data/NIS090.AT2'
)

eql_calc = pystrata.propagation.EquivalentLinearCalculator(strain_limit = 0.5)

In [9]:
p = discretized_Site_profile.copy()

eql_calc(motion, #type:ignore
        p,
        p.location("outcrop", index=-1))

In [10]:
output(eql_calc,
        name = f"test",)

In [11]:
FAS_df = output[0].to_dataframe()

FAS = FAS_df.iloc[:,0].values
freq = FAS_df.index.to_numpy()

In [12]:
kappa_verify = calc_kappa(freq,FAS,Kappa_freqs,None)
kappa = output[1].values

In [13]:
print(kappa_verify)
print(kappa)

0.48825322811228866
[0.48825323]


In [14]:
kappa_verify = calc_kappa(freq,FAS,Kappa_freqs,None)
kappa = output[1].values

In [15]:
delta_kappa = kappa - 0.039

In [16]:
fas_kappa_verify = Kappa_Correction(freq,FAS,delta_kappa)

fas_kappa = output[2].values

error = fas_kappa - fas_kappa_verify

In [17]:
print(error)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.